# BigQuery exploration in the sage-baker project

Two ways to read BQ data: the `%%bigquery` cell magic (ergonomic) and the
direct `google.cloud.bigquery` Python client (more control).

The kernel needs to be **Python (sage-baker)** — pick it from the kernel
switcher in the top right if it isn't already selected.

Prereqs:
- `make install-jupyter` once (gets you `bigquery-magics`)
- `.env` with `GOOGLE_APPLICATION_CREDENTIALS` and `GOOGLE_CLOUD_PROJECT`
- `make bq-upload-sonar` once if you want to query the sonar table
- Launch via `make jupyter` so `.env` is exported to the kernel

In [ ]:
# project setup — chdir to project root so relative paths work, src/ on sys.path
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'src')

%load_ext autoreload
%autoreload 2
%load_ext bigquery_magics

PROJECT = os.environ.get('GOOGLE_CLOUD_PROJECT')
print('GOOGLE_CLOUD_PROJECT =', PROJECT)
print('cwd =', os.getcwd())

## Pattern 1: `%%bigquery` cell magic

Result lands in the named variable as a `pandas.DataFrame`.

In [ ]:
%%bigquery sonar_df --project $PROJECT
SELECT * FROM `gen-lang-client-0749458129.sage_baker.sonar` LIMIT 10

In [ ]:
sonar_df.shape, sonar_df['target'].value_counts().to_dict()

## Pattern 2: direct `google.cloud.bigquery` client

More control — useful for parameterized queries, loading larger
datasets via the Storage API, or building dynamic SQL.

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

df = client.query(f"""
    SELECT *
    FROM `{PROJECT}.sage_baker.sonar`
""").to_dataframe()

print(f'{len(df)} rows × {len(df.columns)} cols')
df.describe().T.head(8)

## Train on it

The trainers in `src/` accept any directory containing a CSV or parquet.
We can write the BQ result to a parquet and call the trainer directly.

In [ ]:
import bundle, train, importlib
importlib.reload(train)

df.to_parquet('data/training.parquet', index=False)
sys.argv = ['train.py', '--train', 'data', '--model-dir', 'model_nb']
train.main()

In [ ]:
# inspect the bundle metadata — including the lineage if data/lineage.json existed
import json
json.loads(open('model_nb/metadata.json').read())

## Inference test

In [ ]:
model = train.model_fn('model_nb')
X = df.drop(columns=['target']).head(5)
y = df['target'].head(5).tolist()
preds = model.predict(X).tolist()
print('actual:   ', y)
print('predicted:', preds)